# Признаки из механики настоящих ботов

До сих пор признаки выводились из здравого смысла: «человек отвлекается, робот работает ровно».
Здесь другой заход — посмотреть, **как скраперы написаны на самом деле**, и вывести признаки из
их кода.

Материал: рабочий сборщик из соседнего проекта (`MechkaloShop/avito_collect.py` и
`parser/sites/avito.py`) плюс публичные репозитории парсеров Авито.

## Что делает настоящий сборщик

Ключевая строка из главного цикла:

```python
await asyncio.sleep(per * random.uniform(0.6, 1.4))
```

Это «очеловечивание» паузы: базовая задержка `per` умножается на случайный множитель из отрезка
от 0.6 до 1.4. Ровно тот приём, который пишут все, кто не хочет попасть под простой лимит
«N запросов в секунду».

Остальное поведение из того же кода:

| что делает | как выглядит в логах |
|---|---|
| один HTTP-запрос на задачу, карточки не открывает | много выдачи, мало просмотров |
| идёт по списку запросов подряд, не возвращается | нет повторных обращений к тем же объявлениям |
| **фиксированные cookie и user-agent** на весь прогон | одна кука, один UA |
| останавливается после трёх блокировок подряд | резкий обрыв активности |
| `httpx` без браузера | нет событий с координатами мыши |

## Что из этого следует математически

Если пауза равна `per · U(0.6, 1.4)`, распределение интервалов между событиями:

- **ограничено с обеих сторон** — всё лежит в коридоре `[0.6·per, 1.4·per]`, то есть ±40% вокруг
  среднего. Ни одного интервала короче или длиннее;
- имеет **постоянный коэффициент вариации** ≈ 0.231, независимо от того, насколько быстро бот
  работает;
- даёт отношение 10-го процентиля к 90-му ≈ 0.48, и это тоже не зависит от скорости.

У человека распределение совсем другой формы — тяжелохвостое: пара кликов подряд за доли
секунды, потом пауза в несколько минут, потом снова серия.

Нарисуем обе формы рядом с тем, что реально лежит в данных.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

import avito_lib as L
from metric import precision_at_recall

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

train, test, events = L.load_data()
ytr = train["target"].values.astype(int)
lab = train.set_index("cookie_id")["target"]
ev = L.clip_to_window(events, train)
ev["dt"] = ev.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
d = ev.dropna(subset=["dt"])

In [ ]:
rng = np.random.default_rng(SEED)
sim = 30 * rng.uniform(0.6, 1.4, 5000)

real_h = d.loc[d.cookie_id.map(lab) == 0, "dt"].values
real_b = d.loc[d.cookie_id.map(lab) == 1, "dt"].values

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].hist(sim, bins=40, color="#c0392b")
ax[0].set_title("Симуляция: sleep(30 * uniform(0.6, 1.4))")
ax[0].set_xlabel("интервал, с")

for v, nm, c in [(real_h, "люди", "#2e86c1"), (real_b, "боты", "#c0392b")]:
    ax[1].hist(np.log10(np.clip(v, 0.1, None)), bins=60, alpha=0.55, density=True, label=nm, color=c)
ax[1].set_title("Данные: интервалы в логарифмической шкале")
ax[1].set_xlabel("log10(интервал, с)")
ax[1].legend()
plt.tight_layout()
plt.show()

print("симуляция бота:  cv =", round(sim.std() / sim.mean(), 3),
      " p10/p90 =", round(np.percentile(sim, 10) / np.percentile(sim, 90), 3))
print("теория для U(0.6,1.4):  cv = 0.231   p10/p90 = 0.48")

Слева — то, что порождает настоящий бот: узкий бугор без хвостов. Справа реальные данные в
логарифмической шкале: у обеих групп разброс на порядки, но у ботов масса смещена и хвост короче.

Важно: **бот в данных не такой чистый, как симуляция.** Он не единственный источник событий у
куки, да и датасет синтетический. Поэтому ищем не точное совпадение с коридором, а сдвиг формы
в его сторону.

## Публичные репозитории

Чтобы не опираться на один пример, посмотрел, что вообще пишут. Самый популярный парсер Авито —
[Duff89/parser_avito](https://github.com/Duff89/parser_avito), 738 звёзд, Python + Playwright.
Рядом ещё десяток: [подборка по тегу avito-parser](https://github.com/topics/avito-parser).

Что у них общего по описаниям:

- **постраничный обход выдачи** — идут по страницам 2, 3, 4 подряд;
- **ротация прокси** для обхода блокировок по IP;
- **джиттер задержки** — тот же `time.sleep(random.uniform(min, max))`, это самый
  распространённый способ;
- часть работает через браузер (Playwright, Selenium), часть — голым `requests`.

Последнее важно для нас: если бот ходит через `requests` без браузера, **событий с координатами
указателя у него не будет вовсе**. Это объясняет, почему группа «Была ли мышь» дала самый
большой прирост из всех.

Из описаний не видно конкретных таймингов — README рассказывают про возможности, а не про
внутренности. Поэтому основой остаётся разобранный код, а репозитории подтверждают, что приёмы
типовые.

## Группа «Механика бота»

Семь признаков, каждый выведен из конкретной особенности кода, а не подобран перебором.

| признак | что считает | откуда взялся |
|---|---|---|
| `dt_p10_p90` | 10-й процентиль интервалов делить на 90-й | джиттер ограничен с обеих сторон |
| `dt_in_corridor` | доля интервалов в коридоре ±40% от медианы | прямая проверка на `uniform(0.6, 1.4)` |
| `dt_iqr_over_median` | межквартильный размах к медиане | робастный аналог разброса, не боится выбросов |
| `dt_std_over_range` | std делить на (max − min) | форма распределения: у равномерного ≈ 0.289 |
| `dt_min_over_median` | минимальный интервал к медианному | у бота жёсткий нижний порог, человек иногда кликает мгновенно |
| `dt_autocorr` | автокорреляция соседних интервалов, лаг 1 | независимый джиттер даёт ноль, у человека серии |
| `view_per_query` | карточек открыто на один уникальный запрос | сборщик листает много страниц по одному запросу |

Отдельно проверено и **не включено**: `search_per_query` (AUC 0.702) оказался монотонным
преобразованием уже имеющегося `q_repeat_frac` — новой информации ноль. Так же слабы
`view_per_search`, `phone_per_view` и флаг «искал, но ничего не открыл».

In [ ]:
def build_mechanics(meta, events_raw):
    ev = L.clip_to_window(events_raw, meta)
    idx = pd.Index(meta["cookie_id"].values, name="cookie_id")
    ev = ev.copy()
    ev["dt"] = ev.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
    dd = ev.dropna(subset=["dt"])
    g = dd.groupby("cookie_id")["dt"]

    p10, p50, p90 = g.quantile(.10), g.median(), g.quantile(.90)
    q25, q75 = g.quantile(.25), g.quantile(.75)
    mn, mx, sd = g.min(), g.max(), g.std()

    F = pd.DataFrame(index=idx)
    F["dt_p10_p90"] = (p10 / p90.clip(lower=1e-9)).reindex(idx)
    F["dt_iqr_over_median"] = ((q75 - q25) / p50.clip(lower=1e-9)).reindex(idx)
    F["dt_std_over_range"] = (sd / (mx - mn).clip(lower=1e-9)).reindex(idx)
    F["dt_min_over_median"] = (mn / p50.clip(lower=1e-9)).reindex(idx)

    med = dd["cookie_id"].map(p50)
    inside = ((dd["dt"] >= 0.6 * med) & (dd["dt"] <= 1.4 * med)).astype(float)
    F["dt_in_corridor"] = inside.groupby(dd["cookie_id"]).mean().reindex(idx)

    F["dt_autocorr"] = g.apply(lambda s: s.autocorr(1) if len(s) > 3 else np.nan).reindex(idx)

    srch = ev[ev["event_name"] == "search_results_view"]
    nq = srch.groupby("cookie_id")["search_query"].nunique()
    nv = ev[ev["event_name"] == "item_view"].groupby("cookie_id").size()
    F["view_per_query"] = (nv.reindex(idx).fillna(0) / nq.reindex(idx).clip(lower=1))

    return F.reset_index(drop=True)


M_tr = build_mechanics(train, events)
MECH = list(M_tr.columns)
print(M_tr.shape, MECH)

In [ ]:
day = train["window_start_ts"].dt.day.values
early, late = day <= 12, day >= 13

pop = L.build_population_stats(events, train, test)
Xtr, GROUPS = L.build_features(train, events, pop)
BASE83 = [c for k, v in GROUPS.items() if k != "sequence" for c in v]

rows = []
for c in MECH:
    x = M_tr[c].astype(float)
    xf = x.fillna(x.median())
    a_e = roc_auc_score(ytr[early], xf[early])
    a_l = roc_auc_score(ytr[late], xf[late])
    corr = Xtr[BASE83].corrwith(x).abs().max()
    rows.append({"признак": c, "AUC ранние дни": round(a_e, 4), "AUC поздние дни": round(a_l, 4),
                 "сила": round(abs(a_l - .5), 4), "стабилен": np.sign(a_e - .5) == np.sign(a_l - .5),
                 "заполнен": round(x.notna().mean(), 3),
                 "макс. корр. с имеющимися": round(corr, 3)})
display(pd.DataFrame(rows).sort_values("сила", ascending=False).reset_index(drop=True))

In [ ]:
pp = M_tr["dt_p10_p90"].copy()
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for t, nm, c in [(0, "люди", "#2e86c1"), (1, "боты", "#c0392b")]:
    ax[0].hist(pp[ytr == t].dropna(), bins=50, alpha=0.55, density=True, label=nm, color=c)
ax[0].set_xlabel("p10 / p90 интервалов")
ax[0].set_title("Насколько зажаты интервалы")
ax[0].legend()

cor = M_tr["dt_in_corridor"]
for t, nm, c in [(0, "люди", "#2e86c1"), (1, "боты", "#c0392b")]:
    ax[1].hist(cor[ytr == t].dropna(), bins=40, alpha=0.55, density=True, label=nm, color=c)
ax[1].set_xlabel("доля интервалов в коридоре ±40% от медианы")
ax[1].set_title("Прямая проверка на uniform(0.6, 1.4)")
ax[1].legend()
plt.tight_layout()
plt.show()

print("медианы:")
display(M_tr.assign(t=ytr).groupby("t")[MECH].median().round(4))

## Решающая проверка

Одномерный AUC пользы **не доказывает**. `dt_p10` и `dt_p90` уже есть в модели по отдельности, и
бустинг мог частично выучить их отношение сам. Единственный честный тест — прогон
`83 признака` против `83 + механика` по обеим схемам валидации.

Условия приёма записаны заранее и не меняются:

1. средняя разница положительна;
2. выигрыш минимум в 90% bootstrap-пересборок;
3. знак не меняется ни на одном из трёх повторов кросс-валидации;
4. подтверждается на хронологическом протоколе.

In [ ]:
X2 = pd.concat([Xtr, M_tr], axis=1)

folds_r = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED).split(Xtr, ytr))
dayn = train["window_start_ts"].dt.normalize()
SPEC = [("2026-04-06", "2026-04-13", "2026-04-14", "2026-04-15"),
        ("2026-04-06", "2026-04-15", "2026-04-16", "2026-04-17"),
        ("2026-04-06", "2026-04-17", "2026-04-18", "2026-04-19")]
folds_t = [(np.where(((dayn >= a) & (dayn <= b)).values)[0],
            np.where(((dayn >= c) & (dayn <= e)).values)[0]) for a, b, c, e in SPEC]


def mk():
    return XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
                         colsample_bytree=0.8, tree_method="hist", eval_metric="logloss",
                         random_state=SEED, verbosity=0, n_jobs=-1)


def run(X, cols, folds, n_rep=1):
    reps = np.full((n_rep, len(ytr)), np.nan)
    per = len(folds) // n_rep
    for k, (tr, va) in enumerate(folds):
        m = mk()
        m.fit(X.iloc[tr][cols], ytr[tr])
        reps[k // per, va] = m.predict_proba(X.iloc[va][cols])[:, 1]
    return np.nanmean(reps, axis=0), reps


def sc(p):
    s = ~np.isnan(p)
    return precision_at_recall(ytr[s], p[s])


a_r, a_rep = run(Xtr, BASE83, folds_r, 3)
b_r, b_rep = run(X2, BASE83 + MECH, folds_r, 3)
a_t, _ = run(Xtr, BASE83, folds_t)
b_t, _ = run(X2, BASE83 + MECH, folds_t)
print(f"83 признака:        случайный {sc(a_r):.4f}   хронологический {sc(a_t):.4f}")
print(f"83 + механика (90): случайный {sc(b_r):.4f}   хронологический {sc(b_t):.4f}")

In [ ]:
def paired(a, b, n=500):
    m = ~(np.isnan(a) | np.isnan(b))
    y_, a_, b_ = ytr[m], a[m], b[m]
    r = np.random.default_rng(SEED)
    out = np.empty(n)
    for i in range(n):
        ix = r.integers(0, len(y_), len(y_))
        out[i] = precision_at_recall(y_[ix], b_[ix]) - precision_at_recall(y_[ix], a_[ix])
    return out


dd_ = paired(a_r, b_r)
reps = [sc(b_rep[i]) - sc(a_rep[i]) for i in range(3)]
d_time = sc(b_t) - sc(a_t)

cond = {
    "1. средняя разница > 0": dd_.mean() > 0,
    "2. выигрыш в >= 90% пересборок": (dd_ > 0).mean() >= 0.90,
    "3. знак одинаков на всех повторах": all(x > 0 for x in reps),
    "4. подтверждено хронологией": d_time > 0,
}
print(f"разница метрики: {sc(b_r) - sc(a_r):+.4f}")
print(f"bootstrap: среднее {dd_.mean():+.4f}, положительна в {(dd_ > 0).mean():.1%} пересборок")
print(f"по повторам CV: {', '.join(f'{x:+.4f}' for x in reps)}")
print(f"хронологический протокол: {d_time:+.4f}")
print()
for k, v in cond.items():
    print(f"  [{'OK' if v else '--'}] {k}")
print()
print("ВЕРДИКТ:", "ГРУППА ПРИНЯТА" if all(cond.values()) else "группа отклонена")

## Вывод

Если группа принята — добавляем её в `avito_lib.py` как `mechanics`, и `solution_final.ipynb`
подхватит её автоматически: там отбор идёт циклом по `GROUPS`.

Если отклонена — не переносим. Это тоже результат: гипотеза была выведена из механики реального
кода, проверена по заранее записанным условиям и не подтвердилась. Бустинг уже вытащил из
`dt_p10` и `dt_p90` всё, что там было.

Что в любом случае стоит унести из этого разбора: **самая результативная группа признаков во всём
решении — «Была ли мышь»** (+0.14). Разбор кода объясняет почему: значительная часть парсеров
ходит через `requests` или `httpx` без браузера, и координат указателя у них не появляется в
принципе. Не потому что бот их прячет, а потому что мыши нет.